In [ ]:
import numpy as np
import math
import plotly.graph_objects as go

import sys
sys.path.append('..')


import numpy as np
import math
import plotly.graph_objects as go


def cylinder_points_local(center_xy, z_center, R_local, H_local, L, R, H):
    """
    - center_xy, z_center: center of local cylinder
    - R_local, H_local: half-sizes of local cylinder
    - L: nearest-neighbor spacing
    - R, H: global cylinder dimensions
    """
    cx, cy = center_xy
    cz = z_center

    # in-plane triangular lattice spacing
    dx = L
    dy = math.sqrt(3) / 2 * L
    dz = math.sqrt(3) / 2 * L #math.sqrt(2.0 / 3.0) * L  # vertical spacing

    # bounds of local search region
    x_min, x_max = cx - R_local, cx + R_local
    y_min, y_max = cy - R_local, cy + R_local
    z_min, z_max = cz - H_local, cz + H_local

    xs = np.arange(x_min, x_max + dx, dx)
    ys = np.arange(y_min, y_max + dy, dy)
    zs = np.arange(z_min, z_max + dz, dz)

    pts = []
    for i, z in enumerate(zs):
        z_global = z
        if z_global < -H/2 or z_global > H/2:
            continue

        layer_shift_x = 0.0
        layer_shift_y = 0.0
        if i % 2 == 1:
            layer_shift_x = 0.5 * dx
            layer_shift_y = 0.5 * dy

        for ix, x in enumerate(xs):
            for iy, y in enumerate(ys):
                xg = x + (iy % 2) * 0.5 * dx + layer_shift_x
                yg = y + layer_shift_y

                # check global cylinder constraint
                if (xg**2 + yg**2) <= R**2:
                    pts.append((xg, yg, z_global))

    return np.array(pts)


def hierarchical_pos_cylinder_search(target, R, H, L0=6.0, levels=6, reduction=0.5,
                            min_L=0.05, min_local_factor=1.5):
    """
    Hierarchical pos search
    Shrinks local cylinder size after each refinement using distance bound.
    """
    cx, cy, cz = target
    center_xy = (0.0, 0.0)     # global center
    z_center = 0
    R_local = R
    H_local = H / 2.0

    L = L0
    path = []

    for lvl in range(levels):
        if L < min_L:
            break

        pts = cylinder_points_local(center_xy, z_center, R_local, H_local, L, R, H)
        if pts.size == 0:
            break

        dists = np.linalg.norm(pts - np.array(target)[None, :], axis=1)
        best_idx = int(np.argmin(dists))
        best_pt = pts[best_idx]
        best_dist = float(dists[best_idx])

        path.append({
            "level": lvl,
            "L": L,
            "num_points": len(pts),
            "best_point": best_pt,
            "best_dist": best_dist,
            "center_xy": center_xy,
            "z_center": z_center,
            "R_local": R_local,
            "H_local": H_local
        })

        # prepare next refinement
        center_xy = (best_pt[0], best_pt[1])
        z_center = best_pt[2]

        L_next = L * reduction
        R_local = math.sqrt(3) / 2 * L/2
        H_local = math.sqrt(3) / 2 * L/2

        L = L_next

    return path


# -------------------------------
# Demo + Visualization
# -------------------------------
if __name__ == "__main__":
    R, H = 12, 24
    target = (-6.3, -5.1, 7.4)

    path = hierarchical_pos_cylinder_search(target, R, H, L0=6.0, levels=3)

    # print results
    for step in path:
        lvl = step["level"]
        print(f"Level {lvl}: L={step['L']:.4f}, num_pts={step['num_points']}, "
              f"best_pt={step['best_point']}, dist={step['best_dist']:.4f}")

    # visualize with Plotly
    fig = go.Figure()

    colors = ["red", "orange", "green", "blue", "purple", "black"]

    for i, step in enumerate(path):
        pts = cylinder_points_local(
            step["center_xy"], step["z_center"],
            step["R_local"], step["H_local"],
            step["L"], R, H
        )
        fig.add_trace(go.Scatter3d(
            x=pts[:,0], y=pts[:,1], z=pts[:,2],
            mode="markers",
            marker=dict(size=6, color=colors[i % len(colors)], opacity=0.5),
            name=f"Level {i}"
        ))
        bp = step["best_point"]
        fig.add_trace(go.Scatter3d(
            x=[bp[0]], y=[bp[1]], z=[bp[2]],
            mode="markers",
            marker=dict(size=6, color=colors[i % len(colors)], symbol="x"),
            name=f"Best L{i}"
        ))

    # target point
    fig.add_trace(go.Scatter3d(
        x=[target[0]], y=[target[1]], z=[target[2]],
        mode="markers",
        marker=dict(size=6, color="black", symbol="diamond"),
        name="Target"
    ))

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[-R, R]),
            yaxis=dict(range=[-R, R]),
            zaxis=dict(range=[-H/2, H/2]),
        ),
        title="Grid Search in Cylinder",
        showlegend=True
    )

    fig.show()